# ReefScan-ID: Stage 2 baseline (E2)

A frozen ResNet50 with logistic regression, compared with the fine-tuned EfficientNet-B0 of notebook 05. The baseline is trained on the Coralscapes patches only and scored on Coralscapes validation. The test split is not used. Run in Google Colab with a GPU runtime.

### Setup: Colab runtime and Google Drive

In [ ]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU: {}".format(gpu_name))
print("CPU count: {}".format(os.cpu_count()))

base_dir = "/content/drive/MyDrive/ReefScan-ID"
data_dir = base_dir + "/Processed/stage2_patches"
print("Drive folder found: {}".format(os.path.isdir(base_dir)))
print("Patch folder found: {}".format(os.path.isdir(data_dir)))

### Setup: libraries

In [ ]:
import random
import time
import numpy as np
import pandas as pd
import sklearn
import torchvision

print("torch version: {}".format(torch.__version__))
print("torchvision version: {}".format(torchvision.__version__))
print("scikit-learn version: {}".format(sklearn.__version__))
print("numpy version: {}".format(np.__version__))
print("pandas version: {}".format(pd.__version__))

### Settings

In [ ]:
loader_crop = 128
patch_size = 224
batch_size = 64
num_workers = 2
seed = 0
resnet_weights_name = "IMAGENET1K_V1"
c_grid = [0.0001, 0.001, 0.01, 0.1, 1.0]
max_iter = 1000

state_names = {0: "healthy", 1: "bleached", 2: "dead"}
features_dir = base_dir + "/Processed/stage2_features/resnet50"
ckpt_path = base_dir + "/Checkpoints/stage2_efficientnet_b0/coralscapes/last.pt"

# Mean and std printed by notebook 05 for its EfficientNet-B0 weights
notebook05_mean = [0.485, 0.456, 0.406]
notebook05_std = [0.229, 0.224, 0.225]

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

print("Loader crop: {} px, resized to {} px, no augmentation".format(loader_crop, patch_size))
print("Feature batch size: {}, workers: {}".format(batch_size, num_workers))
print("ResNet50 weights: {}".format(resnet_weights_name))
print("Grid for C: {}, max iterations: {}".format(c_grid, max_iter))
print("Feature folder: {}".format(features_dir))
print("EfficientNet-B0 checkpoint: {}".format(ckpt_path))

## Section 1: E2 baseline

Question: how does a logistic regression on frozen ResNet50 features compare with the fine-tuned EfficientNet-B0 on the Coralscapes validation set, when both learn from the same Coralscapes train patches?

Checks: load the saved Coralscapes patches and print the rows per class, extract the pooled ResNet50 features once and cache them on Drive, fit a logistic regression for a small grid of C and pick C on Coralscapes validation macro-F1, then print macro-F1, per-class recall and the confusion matrix of the baseline next to the best epoch of the EfficientNet-B0 run. Only the train and validation splits are used.

### Saved patch files and rows per class

In [ ]:
cs_manifest = pd.read_csv(data_dir + "/coralscapes_manifest.csv")
print("Coralscapes manifest columns: {}".format(list(cs_manifest.columns)))
print("Splits in the manifest: {}".format(sorted(cs_manifest["split"].unique())))

for split in ["train", "validation"]:
    array_path = "{}/coralscapes_{}_{}.npy".format(data_dir, split, loader_crop)
    print("{}: {:.1f} MB".format(array_path.split("/")[-1], os.path.getsize(array_path) / 1024 ** 2))

print("Rows per split and label:")
print(pd.crosstab(cs_manifest["split"], cs_manifest["label"]).to_string())

# Counts printed by notebooks 04 and 05
expected = {"train": {"healthy": 18060, "bleached": 840, "dead": 3353},
            "validation": {"healthy": 1437, "bleached": 106, "dead": 777}}
for split, counts in expected.items():
    found = cs_manifest[cs_manifest["split"] == split]["label"].value_counts().to_dict()
    print("{}: {} rows, expected {}, same as expected: {}".format(
        split, sum(found.values()), sum(counts.values()), found == counts))

### Datasets and loaders

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


class PatchDataset(Dataset):
    def __init__(self, split):
        table = cs_manifest[cs_manifest["split"] == split]
        self.table = table.reset_index(drop=True)
        self.rows = self.table["row"].to_numpy()
        self.states = self.table["state"].to_numpy()
        self.array_path = "{}/coralscapes_{}_{}.npy".format(data_dir, split, loader_crop)
        self.patches = None

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        # The array file is opened on first use
        if self.patches is None:
            self.patches = np.load(self.array_path, mmap_mode="r")
        patch = torch.from_numpy(np.array(self.patches[self.rows[index]]))
        image = patch.permute(2, 0, 1).float().div(255)
        if image.shape[-1] != patch_size:
            image = F.interpolate(image.unsqueeze(0), size=(patch_size, patch_size),
                                  mode="bilinear", align_corners=False).squeeze(0)
        return image, int(self.states[index]), 0


# The order is not shuffled, so features and labels stay in manifest order
train_data = PatchDataset("train")
val_data = PatchDataset("validation")
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=False, num_workers=num_workers)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=num_workers)

y_train = train_data.states
y_val = val_data.states
print("Train rows: {}, batches: {}".format(len(train_data), len(train_loader)))
print("Validation rows: {}, batches: {}".format(len(val_data), len(val_loader)))
print("Train labels (healthy, bleached, dead): {}".format(np.bincount(y_train, minlength=3).tolist()))
print("Validation labels (healthy, bleached, dead): {}".format(np.bincount(y_val, minlength=3).tolist()))

### ResNet50 backbone

In [ ]:
from torch import nn
from torchvision.models import ResNet50_Weights, resnet50

weights = ResNet50_Weights[resnet_weights_name]
backbone = resnet50(weights=weights)
backbone.fc = nn.Identity()
backbone = backbone.to(device).eval()
for parameter in backbone.parameters():
    parameter.requires_grad = False

transforms = weights.transforms()
image_mean = torch.tensor(transforms.mean, device=device).view(1, 3, 1, 1)
image_std = torch.tensor(transforms.std, device=device).view(1, 3, 1, 1)


def normalise(images):
    return (images - image_mean) / image_std


print("Weights: {}".format(weights))
print("Transforms that come with the weights: {}".format(transforms))
print("Mean {}, std {}".format(transforms.mean, transforms.std))
print("Same mean and std as notebook 05: {}".format(
    list(transforms.mean) == notebook05_mean and list(transforms.std) == notebook05_std))
print("Backbone parameters: {}, trainable: {}".format(
    sum(p.numel() for p in backbone.parameters()), sum(p.numel() for p in backbone.parameters() if p.requires_grad)))

### Features, cached on Drive

In [ ]:
os.makedirs(features_dir, exist_ok=True)


def feature_path(split):
    return "{}/coralscapes_{}_{}to{}_{}.npy".format(
        features_dir, split, loader_crop, patch_size, resnet_weights_name.lower())


def extract(loader):
    output = np.zeros((len(loader.dataset), 2048), dtype=np.float32)
    start = 0
    with torch.no_grad():
        for images, labels, batch_sources in loader:
            pooled = backbone(normalise(images.to(device)))
            output[start:start + len(pooled)] = pooled.cpu().numpy()
            start += len(pooled)
    return output


features = {}
for split, loader in [("train", train_loader), ("validation", val_loader)]:
    file_path = feature_path(split)
    start = time.time()
    if os.path.isfile(file_path):
        features[split] = np.load(file_path)
        print("{}: reused {}, loaded in {:.1f} seconds".format(split, file_path.split("/")[-1], time.time() - start))
    else:
        features[split] = extract(loader)
        seconds = time.time() - start
        np.save(file_path, features[split])
        print("{}: extracted and saved {}".format(split, file_path.split("/")[-1]))
        print("{}: {:.1f} seconds, {:.0f} patches per second".format(
            split, seconds, len(features[split]) / seconds))

    print("{}: features shape {}, dtype {}".format(split, features[split].shape, features[split].dtype))
    print("{}: rows equal the manifest rows: {}".format(split, len(features[split]) == len(loader.dataset)))
    print("{}: all values finite: {}".format(split, bool(np.isfinite(features[split]).all())))

### Scoring function

In [ ]:
def confusion_scores(confusion):
    # Same formulas as notebook 05, section 4
    confusion = np.array(confusion)
    true_count = confusion.sum(axis=1)
    predicted_count = confusion.sum(axis=0)
    correct = np.diag(confusion)
    present = true_count > 0
    recall = correct / np.maximum(true_count, 1)
    precision = correct / np.maximum(predicted_count, 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    return recall, present, float(f1[present].mean())


def confusion_matrix(y_true, y_pred):
    matrix = np.zeros((len(state_names), len(state_names)), dtype=np.int64)
    np.add.at(matrix, (y_true, y_pred), 1)
    return matrix


print("Scoring functions defined: confusion_scores returns recall per class, classes present, macro-F1")

### Standardise the features

In [ ]:
from sklearn.preprocessing import StandardScaler

# The scaler learns the mean and std from the train features only
scaler = StandardScaler().fit(features["train"])
x_train = scaler.transform(features["train"])
x_val = scaler.transform(features["validation"])

print("Train features: {}, validation features: {}".format(x_train.shape, x_val.shape))
print("Features with zero variance in train: {}".format(int((scaler.var_ == 0).sum())))
print("Train mean of the standardised features: {:.4f}, std: {:.4f}".format(x_train.mean(), x_train.std()))
print("Validation mean of the standardised features: {:.4f}, std: {:.4f}".format(x_val.mean(), x_val.std()))

### Logistic regression over a grid of C

In [ ]:
from sklearn.linear_model import LogisticRegression

fits = {}
grid_rows = []
for c in c_grid:
    start = time.time()
    classifier = LogisticRegression(C=c, class_weight="balanced", max_iter=max_iter)
    classifier.fit(x_train, y_train)
    confusion = confusion_matrix(y_val, classifier.predict(x_val))
    recall, present, macro_f1 = confusion_scores(confusion)
    seconds = time.time() - start

    fits[c] = {"confusion": confusion, "macro_f1": macro_f1}
    row = {"C": c, "macro-F1": macro_f1}
    for class_id, class_name in state_names.items():
        row["recall " + class_name] = recall[class_id]
    row["iterations"] = int(classifier.n_iter_[0])
    row["converged"] = bool(classifier.n_iter_[0] < max_iter)
    row["seconds"] = seconds
    grid_rows.append(row)
    print("C {}: validation macro-F1 {:.4f}, {} iterations, {:.0f} seconds".format(
        c, macro_f1, row["iterations"], seconds))

print("Grid results on Coralscapes validation (class_weight balanced, standardised features):")
print(pd.DataFrame(grid_rows).round(4).to_string(index=False))

best_c = max(c_grid, key=lambda c: fits[c]["macro_f1"])
print("C chosen by Coralscapes validation macro-F1: {}".format(best_c))
print("The chosen C is at the edge of the grid: {}".format(best_c in (c_grid[0], c_grid[-1])))

### E2 table: logistic regression against EfficientNet-B0

In [ ]:
import datetime

stamp = datetime.datetime.fromtimestamp(os.path.getmtime(ckpt_path)).strftime("%Y-%m-%d %H:%M")
print("Checkpoint {}: {:.1f} MB, modified {}".format(
    ckpt_path.split("/")[-2] + "/" + ckpt_path.split("/")[-1], os.path.getsize(ckpt_path) / 1024 ** 2, stamp))
print("This cell only reads the checkpoint")

last_state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
history = last_state["history"]

# Same rule as notebook 05, section 4
picked = max(history, key=lambda record: confusion_scores(record["confusions"]["coralscapes"])[2])
effnet_confusion = np.array(picked["confusions"]["coralscapes"])
effnet_f1 = confusion_scores(effnet_confusion)[2]
print("last.pt holds {} epochs from the '{}' run".format(last_state["epoch"], last_state["source"]))
print("Best epoch by Coralscapes validation macro-F1: {} ({:.4f})".format(picked["epoch"], effnet_f1))
print("Reference in the handoff: epoch 11, macro-F1 0.7713; same: {}".format(
    picked["epoch"] == 11 and round(effnet_f1, 4) == 0.7713))
del last_state

models = [
    ("ResNet50 frozen + logistic regression, C {}".format(best_c), fits[best_c]["confusion"]),
    ("EfficientNet-B0 fine-tuned, best epoch {}".format(picked["epoch"]), effnet_confusion),
]
rows = []
for label, confusion in models:
    recall, present, macro_f1 = confusion_scores(confusion)
    row = {"model": label, "macro-F1": macro_f1}
    for class_id, class_name in state_names.items():
        row["recall " + class_name] = recall[class_id]
    rows.append(row)

print("E2 on Coralscapes validation (the test split is not used)")
print(pd.DataFrame(rows).round(3).to_string(index=False))

supports = [np.array(confusion).sum(axis=1).tolist() for label, confusion in models]
print("Validation rows per class (healthy, bleached, dead): {}".format(supports))
print("Same Coralscapes validation set for both models: {}".format(supports[0] == supports[1]))

### Confusion matrices

In [ ]:
class_labels = list(state_names.values())
for label, confusion in models:
    print("{} (rows true, columns predicted)".format(label))
    print(pd.DataFrame(np.array(confusion), index=class_labels, columns=class_labels).to_string())

**Observation**: